# AI Driver Monitoring System: drowsiness-over-time model on Kaggle (UTA-RLDD)

Turns UTA-RLDD videos (60 people × alert / low-vigilant / drowsy) into face-feature sequences with MediaPipe, then trains the
**Conv-Transformer** with **subject-independent 5-fold cross-validation**, plus baselines (GRU, Random Forest) and the
**calibration ablation** (contribution C2).

**Before running**
1. Session options: **Accelerator → None (CPU)** is enough and saves your GPU quota (the work is face tracking on the CPU). **Internet → On.**
2. **Add Input** → the UTA-RLDD dataset (e.g. `thuannn18022005/uta-rldd-full`).
3. **Save Version → Save & Run All**. It takes about 2-4 h; the log shows progress per video.
4. Download `dms_temporal.zip` from Output and unzip it in the project (creates `runs_temporal/…` and `features/rldd/…`).

In [ ]:
REPO = "github.com/Narayani26/AI-Driver-monitoring-system.git"
BRANCH = "feature/dms-implementation"   # or "main" once merged
MAX_SECONDS = 360   # use the first 6 min of each ~10 min video (set None for all; ~2x longer)
WORKERS = 4         # Kaggle sessions have 4 CPU cores
EPOCHS = 20

In [ ]:
import os, subprocess, sys
if not os.path.exists("/kaggle/working/dms"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, f"https://{REPO}", "/kaggle/working/dms"], check=True)
os.chdir("/kaggle/working/dms")
sys.path.insert(0, "/kaggle/working/dms")
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y libegl1 libgles2 > /dev/null 2>&1 || true
!pip install -q mediapipe timm scikit-learn
!git log --oneline -1

## 1. Find the videos

In [ ]:
from pathlib import Path
from dms.data.probe import main as probe
from dms.data.discover import dataset_roots
from dms.data.sequences import find_videos
probe(["/kaggle/input"])
roots = [(len(find_videos(r)), r) for r in dataset_roots(Path("/kaggle/input"))]
n, RLDD = max(roots) if roots else (0, None)
print(f"using {RLDD} ({n} videos)" if n else "NO VIDEOS FOUND - paste the probe output above to get the layout supported")

## 2. Video → face-feature sequences (resumable, parallel)

In [ ]:
if RLDD:
    ms = f"--max-seconds {MAX_SECONDS}" if MAX_SECONDS else ""
    !python -m dms.data.sequences "{RLDD}" features/rldd --fps 10 --workers {WORKERS} {ms}

## 3. Train and compare (subject-independent 5-fold CV)

In [ ]:
if RLDD:
    common = f"--folds 5 --epochs {EPOCHS} --steps-per-epoch 100"
    !python -m dms.training.train_temporal features/rldd --model transformer {common} --fit-all --out runs_temporal/transformer
    !python -m dms.training.train_temporal features/rldd --model gru {common} --out runs_temporal/gru
    !python -m dms.training.train_temporal features/rldd --model rf --folds 5 --out runs_temporal/rf
    !python -m dms.training.train_temporal features/rldd --model transformer {common} --no-calibration --out runs_temporal/transformer_nocalib
    !python -m dms.training.train_temporal features/rldd --model transformer {common} --seq-len 100 --out runs_temporal/transformer_10s

## 4. Results + download

In [ ]:
import json, glob
print(f"{'run':40s} {'window macro-F1':>18s} {'video accuracy':>18s}")
for p in sorted(glob.glob("runs_temporal/*/metrics.json")):
    s = json.load(open(p))["summary"]
    f1, va = s["window_macro_f1"], s["video_accuracy"]
    print(f"{p.split('/')[1]:40s} {f1['mean']:.3f} ± {f1['std']:.3f}     {va['mean']:.3f} ± {va['std']:.3f}")
!cd /kaggle/working/dms && zip -qr /kaggle/working/dms_temporal.zip runs_temporal features && ls -lh /kaggle/working/dms_temporal.zip